In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Wczytanie danych
df = pd.read_csv('../data/train.csv', low_memory=False)

# Podstawowe informacje
print(f"Liczba rekordów: {df.shape[0]}")
print(f"Liczba kolumn: {df.shape[1]}")
print(f"\nPierwsze 5 wierszy:")
df.head()

Liczba rekordów: 100000
Liczba kolumn: 28

Pierwsze 5 wierszy:


,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good


In [3]:
#Filtrowanie - ostatni miesiac na klienta
month_order = ['January', 'February', 'March', 'April', 
               'May', 'June', 'July', 'August']
df['Month'] = pd.Categorical(df['Month'], 
                              categories=month_order, 
                              ordered=True)
df_last = df.sort_values('Month').groupby('Customer_ID').last().reset_index()

print(f"Rekordów po filtrowaniu: {len(df_last)}")
print(f"\nRozkład klas:")
print(df_last['Credit_Score'].value_counts())

Rekordów po filtrowaniu: 12500

Rozkład klas:
Credit_Score
Standard    6485
Poor        3602
Good        2413
Name: count, dtype: int64


In [4]:
# Selekcja wybranych kolumn
selected = ['Age', 'Occupation', 'Annual_Income', 'Monthly_Inhand_Salary',
            'Num_Bank_Accounts', 'Num_Credit_Card', 'Num_of_Loan',
            'Outstanding_Debt', 'Total_EMI_per_month', 'Delay_from_due_date',
            'Num_of_Delayed_Payment', 'Credit_History_Age', 
            'Payment_of_Min_Amount', 'Credit_Utilization_Ratio', 
            'Credit_Mix', 'Monthly_Balance', 'Credit_Score']

df_selected = df_last[selected].copy()

print(f"Wybrane kolumny: {len(selected)-2} cech + target")
print(f"Kształt zbioru: {df_selected.shape}")
print(f"\nTypy danych:")
print(df_selected.dtypes)

Wybrane kolumny: 15 cech + target
Kształt zbioru: (12500, 17)

Typy danych:
Age                             str
Occupation                      str
Annual_Income                   str
Monthly_Inhand_Salary       float64
Num_Bank_Accounts             int64
Num_Credit_Card               int64
Num_of_Loan                     str
Outstanding_Debt                str
Total_EMI_per_month         float64
Delay_from_due_date           int64
Num_of_Delayed_Payment          str
Credit_History_Age              str
Payment_of_Min_Amount           str
Credit_Utilization_Ratio    float64
Credit_Mix                      str
Monthly_Balance                 str
Credit_Score                    str
dtype: object


In [5]:
import re

# Zamiana pseudo-brakow na NaN
df_selected = df_selected.replace(['_______', '_', '!@9#%8'], np.nan)
df_selected = df_selected.replace(r'^__.*__$', np.nan, regex=True)
df_selected = df_selected.replace(r'^_+$', np.nan, regex=True)

# Czyszczenie Age - usuwamy znaki specjalne i konwertujemy na liczbe
df_selected['Age'] = df_selected['Age'].astype(str).str.replace(r'[^0-9]', '', regex=True)
df_selected['Age'] = pd.to_numeric(df_selected['Age'], errors='coerce')
df_selected.loc[df_selected['Age'] < 18, 'Age'] = np.nan
df_selected.loc[df_selected['Age'] > 100, 'Age'] = np.nan

# Czyszczenie Num_of_Loan
df_selected['Num_of_Loan'] = df_selected['Num_of_Loan'].astype(str).str.replace(r'[^0-9]', '', regex=True)
df_selected['Num_of_Loan'] = pd.to_numeric(df_selected['Num_of_Loan'], errors='coerce')
df_selected.loc[df_selected['Num_of_Loan'] < 0, 'Num_of_Loan'] = np.nan
df_selected.loc[df_selected['Num_of_Loan'] > 20, 'Num_of_Loan'] = np.nan

# Czyszczenie Num_of_Delayed_Payment
df_selected['Num_of_Delayed_Payment'] = df_selected['Num_of_Delayed_Payment'].astype(str).str.replace(r'[^0-9]', '', regex=True)
df_selected['Num_of_Delayed_Payment'] = pd.to_numeric(df_selected['Num_of_Delayed_Payment'], errors='coerce')
df_selected.loc[df_selected['Num_of_Delayed_Payment'] < 0, 'Num_of_Delayed_Payment'] = np.nan

# Czyszczenie Outstanding_Debt
df_selected['Outstanding_Debt'] = df_selected['Outstanding_Debt'].astype(str).str.replace(r'[^0-9.]', '', regex=True)
df_selected['Outstanding_Debt'] = pd.to_numeric(df_selected['Outstanding_Debt'], errors='coerce')

# Czyszczenie Monthly_Balance
df_selected['Monthly_Balance'] = df_selected['Monthly_Balance'].astype(str).str.replace(r'[^0-9.]', '', regex=True)
df_selected['Monthly_Balance'] = pd.to_numeric(df_selected['Monthly_Balance'], errors='coerce')

# Konwersja Credit_History_Age z "X Years and Y Months" na liczbe miesiecy
def parse_credit_history(val):
    if pd.isna(val):
        return np.nan
    match = re.search(r'(\d+)\s+Years?\s+and\s+(\d+)\s+Months?', str(val))
    if match:
        return int(match.group(1)) * 12 + int(match.group(2))
    return np.nan

df_selected['Credit_History_Age'] = df_selected['Credit_History_Age'].apply(parse_credit_history)

print("Czyszczenie zakończone.")
print(f"\nTypy danych po czyszczeniu:")
print(df_selected.dtypes)
print(f"\nBrakujące wartości:")
print(df_selected.isnull().sum())

Czyszczenie zakończone.

Typy danych po czyszczeniu:
Age                         float64
Occupation                      str
Annual_Income                   str
Monthly_Inhand_Salary       float64
Num_Bank_Accounts             int64
Num_Credit_Card               int64
Num_of_Loan                 float64
Outstanding_Debt            float64
Total_EMI_per_month         float64
Delay_from_due_date           int64
Num_of_Delayed_Payment      float64
Credit_History_Age            int64
Payment_of_Min_Amount           str
Credit_Utilization_Ratio    float64
Credit_Mix                      str
Monthly_Balance             float64
Credit_Score                    str
dtype: object

Brakujące wartości:
Age                          988
Occupation                   880
Annual_Income                  0
Monthly_Inhand_Salary          0
Num_Bank_Accounts              0
Num_Credit_Card                0
Num_of_Loan                  566
Outstanding_Debt               0
Total_EMI_per_month            0
Del

In [6]:
# Czyszczenie Annual_Income
df_selected['Annual_Income'] = df_selected['Annual_Income'].astype(str).str.replace(r'[^0-9.]', '', regex=True)
df_selected['Annual_Income'] = pd.to_numeric(df_selected['Annual_Income'], errors='coerce')

# Uzupelnianie brakow
# Zmienne liczbowe - mediana
num_cols = ['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 
            'Num_of_Loan', 'Num_of_Delayed_Payment', 
            'Monthly_Balance', 'Credit_History_Age']

for col in num_cols:
    median_val = df_selected[col].median()
    df_selected[col] = df_selected[col].fillna(median_val)

# Zmienne kategoryczne - moda
cat_cols = ['Occupation', 'Credit_Mix']
for col in cat_cols:
    mode_val = df_selected[col].mode()[0]
    df_selected[col] = df_selected[col].fillna(mode_val)

print("Uzupełnianie braków zakończone.")
print(f"\nBrakujące wartości po uzupełnieniu:")
print(df_selected.isnull().sum())

Uzupełnianie braków zakończone.

Brakujące wartości po uzupełnieniu:
Age                         0
Occupation                  0
Annual_Income               0
Monthly_Inhand_Salary       0
Num_Bank_Accounts           0
Num_Credit_Card             0
Num_of_Loan                 0
Outstanding_Debt            0
Total_EMI_per_month         0
Delay_from_due_date         0
Num_of_Delayed_Payment      0
Credit_History_Age          0
Payment_of_Min_Amount       0
Credit_Utilization_Ratio    0
Credit_Mix                  0
Monthly_Balance             0
Credit_Score                0
dtype: int64


In [7]:
from sklearn.preprocessing import LabelEncoder

# Kodowanie Payment_of_Min_Amount (Yes/No/NM)
df_selected['Payment_of_Min_Amount'] = df_selected['Payment_of_Min_Amount'].map({
    'Yes': 1, 
    'No': 0, 
    'NM': 0
})

# Kodowanie Credit_Mix (Good/Standard/Bad)
df_selected['Credit_Mix'] = df_selected['Credit_Mix'].map({
    'Good': 2, 
    'Standard': 1, 
    'Bad': 0
})

# Kodowanie Occupation - Label Encoding
le_occupation = LabelEncoder()
df_selected['Occupation'] = le_occupation.fit_transform(df_selected['Occupation'])

# Kodowanie targetu Credit_Score
df_selected['Credit_Score'] = df_selected['Credit_Score'].map({
    'Good': 2, 
    'Standard': 1, 
    'Poor': 0
})


print(f"\nSprawdzenie unikalnych wartości:")
print(f"Payment_of_Min_Amount: {df_selected['Payment_of_Min_Amount'].unique()}")
print(f"Credit_Mix: {df_selected['Credit_Mix'].unique()}")
print(f"Credit_Score: {df_selected['Credit_Score'].unique()}")
print(f"\nKształt finalnego zbioru: {df_selected.shape}")


Sprawdzenie unikalnych wartości:
Payment_of_Min_Amount: [1 0]
Credit_Mix: [0 1 2]
Credit_Score: [0 1 2]

Kształt finalnego zbioru: (12500, 17)


In [8]:
# Zapis oczyszczonych danych
df_selected.to_csv('../data/train_clean.csv', index=False)

print("Dane zapisane do pliku train_clean.csv")
print(f"\nPodsumowanie finalnego zbioru:")
print(f"Rekordów: {df_selected.shape[0]}")
print(f"Cech: {df_selected.shape[1]-1}")
print(f"\nRozkład klas po przetworzeniu:")
print(df_selected['Credit_Score'].value_counts())
print(f"\nStatystyki opisowe:")
df_selected.describe()

Dane zapisane do pliku train_clean.csv

Podsumowanie finalnego zbioru:
Rekordów: 12500
Cech: 16

Rozkład klas po przetworzeniu:
Credit_Score
1    6485
0    3602
2    2413
Name: count, dtype: int64

Statystyki opisowe:


,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Num_of_Loan,Outstanding_Debt,Total_EMI_per_month,Delay_from_due_date,Num_of_Delayed_Payment,Credit_History_Age,Payment_of_Min_Amount,Credit_Utilization_Ratio,Credit_Mix,Monthly_Balance,Credit_Score
count,12500.000000,12500.000000,1.250000e+04,12500.000000,12500.000000,12500.000000,12500.000000,12500.000000,12500.000000,12500.000000,12500.000000,12500.000000,12500.00000,12500.000000,12500.000000,12500.000000,12500.000000
mean,34.605840,6.951520,1.616206e+05,4188.592303,16.939920,23.172720,3.506240,1426.220376,1488.394291,21.060880,33.046400,224.608720,0.52568,32.349265,1.053760,403.915782,0.904880
std,9.741622,4.159903,1.297842e+06,3180.147611,114.350815,132.005866,2.393061,1155.169458,8561.449910,14.863091,238.692938,99.655759,0.49936,5.156815,0.654602,213.918403,0.687161
min,18.000000,0.000000,7.005930e+03,303.645417,-1.000000,0.000000,0.000000,0.230000,0.000000,-5.000000,0.000000,8.000000,0.00000,20.100770,0.000000,0.382558,0.000000
25%,27.000000,3.000000,1.945333e+04,1624.937917,3.000000,4.000000,2.000000,566.072500,31.496968,10.000000,9.000000,148.000000,0.00000,28.066517,1.000000,270.146750,0.000000
50%,34.000000,7.000000,3.757238e+04,3087.595000,6.000000,5.000000,3.000000,1166.155000,72.887628,18.000000,14.000000,222.000000,1.00000,32.418953,1.000000,339.749646,1.000000
75%,42.000000,10.000000,7.269021e+04,5947.364167,7.000000,7.000000,5.000000,1945.962500,169.634826,28.000000,18.000000,305.000000,1.00000,36.623650,1.000000,473.320888,1.000000
max,56.000000,14.000000,2.383470e+07,15204.633333,1756.000000,1499.000000,18.000000,4998.070000,81971.000000,67.000000,4293.000000,404.000000,1.00000,48.199824,2.000000,1463.792328,2.000000


In [9]:
# Sprawdzenie realnych klientów Good z datasetu
good_clients = df_selected[df_selected['Credit_Score'] == 2].head(3)
print(good_clients[['Age', 'Occupation', 'Annual_Income', 'Monthly_Inhand_Salary',
                     'Num_Bank_Accounts', 'Num_Credit_Card', 'Num_of_Loan',
                     'Outstanding_Debt', 'Total_EMI_per_month', 'Delay_from_due_date',
                     'Num_of_Delayed_Payment', 'Credit_History_Age',
                     'Payment_of_Min_Amount', 'Credit_Utilization_Ratio',
                     'Credit_Mix', 'Monthly_Balance', 'Credit_Score']].to_string())

     Age  Occupation  Annual_Income  Monthly_Inhand_Salary  Num_Bank_Accounts  Num_Credit_Card  Num_of_Loan  Outstanding_Debt  Total_EMI_per_month  Delay_from_due_date  Num_of_Delayed_Payment  Credit_History_Age  Payment_of_Min_Amount  Credit_Utilization_Ratio  Credit_Mix  Monthly_Balance  Credit_Score
5   27.0           6      46951.020            3725.585000                  7                4          0.0            340.22             0.000000                    8                     9.0                 257                      0                 38.682232           1       225.124505             2
8   31.0           5      89064.520            7256.043333                  5                3          1.0            648.36            37.572751                    6                     5.0                 363                      0                 30.574299           1       641.937464             2
11  45.0           0      15989.085            1086.423750                  5           